In [ ]:
# next step:
# re-ranking algorithms - have some mock children users and filter out ppc and downrank pc and ndc
# move to ci/cd instead of zip deployment

In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
from io import BytesIO
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
def blob_to_df(container_client, blob_name):
    client = container_client.get_blob_client(blob_name)
    data = client.download_blob().readall()
    stream = BytesIO(data)
    df = pd.read_csv(stream)
    return df

In [3]:
# authenticate
credential = DefaultAzureCredential()

load_dotenv()

SUBSCRIPTION = os.getenv("SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
WS_NAME = os.getenv("WORKSPACE_NAME")

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WS_NAME,
)

In [4]:
# Verify that the handle works correctly.
ws = ml_client.workspaces.get(WS_NAME)
print(ws.location, ":", ws.resource_group)

uksouth : trustandsafety-cv-sandbox-rg


In [ ]:
# connect to the blob storage, outputs cleared
from azure.ai.ml.entities import AzureBlobDatastore, AccountKeyConfiguration

DATASTORE_NAME = "recsys_database"
ACCOUNT_KEY = os.getenv("ACCOUNT_KEY")

blob_ds = AzureBlobDatastore(
    name=DATASTORE_NAME,
    account_name="asatrustandsafetycv",
    container_name="recsys",
    credentials=AccountKeyConfiguration(account_key=ACCOUNT_KEY)
)

ml_client.datastores.create_or_update(blob_ds)

In [6]:
dependencies_dir = "./dependencies"
os.makedirs(dependencies_dir, exist_ok=True)

In [117]:
%%writefile {dependencies_dir}/conda.yaml
name: model-env
channels:
  - conda-forge
dependencies:
  - python=3.8
  - numpy=1.21.2
  - pip=21.2.4
  - scikit-learn=0.24.2
  - scipy=1.7.1
  - pandas>=1.1,<1.2
  - cloudpickle
  - pip:
    - inference-schema[numpy-support]==1.3.0
    - xlrd==2.0.1
    - mlflow== 2.4.1
    - azureml-mlflow==1.51.0
    - python-dotenv
    - azureml-core
    - azureml-dataset-runtime
    - azure-storage-blob
    

Overwriting ./dependencies/conda.yaml


In [7]:
# create an env
from azure.ai.ml.entities import Environment

custom_env_name = "recsys_app"

pipeline_job_env = Environment(
    name=custom_env_name,
    description="Custom environment for RecSys App pipeline",
    conda_file=os.path.join(dependencies_dir, "conda.yaml"),
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    version="0.2.4", # chang this if conda.yml is changed
)

pipeline_job_env = ml_client.environments.create_or_update(pipeline_job_env)

print(
    f"Environment with name {pipeline_job_env.name} is registered to workspace, the environment version is {pipeline_job_env.version}"
)

Environment with name recsys_app is registered to workspace, the environment version is 0.2.4


In [8]:
data_prep_src_dir = "./scripts/data_prep"
os.makedirs(data_prep_src_dir, exist_ok=True)

In [42]:
%%writefile {data_prep_src_dir}/data_prep.py
import os
import argparse
import pandas as pd
import numpy as np
from io import BytesIO
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
import logging
import mlflow

def score_interactions(interactions_df):
    """
    This method assigns scores to each user-item interaction.
    Negative scores are prioritized when both positive and negative interactions exist.
    """
    score_map = {
            'Viewed': 2,
            'Like': 3,
            'Dislike': -3,
            'Report': -4,
            'Share': 5,
            'Comment': 4
        }

    # Map interaction scores to a new column
    scores = interactions_df.copy()
    scores['InteractionScore'] = scores['interaction_type'].map(score_map).fillna(0)

    # Group by UserID and ContentID, resolving conflicts
    def resolve_scores(group):
        scoreslist = group['InteractionScore'].tolist()
        if any(score < 0 for score in scoreslist):  # If any negative interaction exists
            return max(filter(lambda x: x < 0, scoreslist))  # Return highest negative score
        else:
            return max(scoreslist)  # Return highest positive score

    # Resolve scores for each user-item pair
    scores = scores.groupby(['user_id', 'content_id']).apply(resolve_scores).reset_index()
    scores.rename(columns={0: 'ResolvedScore'}, inplace=True)

    # Keep the structure for future merges
    return scores

def assign_interest_scores(user_df, content_df, resolved_scores_df):
    """
    Assign a score of 1 to user-content pairs based on interest matching.
    Updates the resolved_scores_df for missing scores.
    
    Parameters:
        user_df (pd.DataFrame): User data with 'UserID' and 'liked_cat' columns.
        content_df (pd.DataFrame): Content data with 'ContentID' and 'categories' columns.
        resolved_scores_df (pd.DataFrame): Existing resolved scores with 'UserID', 'ContentID', and 'ResolvedScore'.
        
    Returns:
        pd.DataFrame: Updated resolved_scores_df with new scores assigned for interest matches.
    """
    # Convert the resolved_scores_df to a set of user-content pairs for quick lookup
    existing_pairs = set(zip(resolved_scores_df['user_id'], resolved_scores_df['content_id']))
    
    # List to store new rows
    new_scores = []
    
    # Iterate over all possible user-content pairs
    for _, user_row in user_df.iterrows():
        user_id = user_row['user_id']
        liked_cat = set(user_row['liked_cat'].split(','))  # Assuming liked_cat is comma-separated
        
        for _, content_row in content_df.iterrows():
            content_id = content_row['content_id']
            categories = set(content_row['categories'].split(','))  # Assuming categories is comma-separated
            
            # Check if the pair already has a score
            if (user_id, content_id) not in existing_pairs:
                # Check for an interest match
                if liked_cat & categories:  # Intersection of sets
                    new_scores.append({'user_id': user_id, 'content_id': content_id, 'ResolvedScore': 1})
    
    # Create a DataFrame for new scores and append to the existing resolved_scores_df
    new_scores_df = pd.DataFrame(new_scores)
    updated_scores_df = pd.concat([resolved_scores_df, new_scores_df], ignore_index=True)
    
    return updated_scores_df


def main():
    """Main function of the script."""

    # input and output arguments
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--interactions", 
        type=str, 
        help="path to input data")
    parser.add_argument(
        "--processed_interactions", 
        type=str, 
        required=True, 
        help="Processed interaction data")
    args = parser.parse_args()

    # Start Logging
    mlflow.start_run()
    print(" ".join(f"{k}={v}" for k, v in vars(args).items()))
    print("input data:", args.interactions)

    interactions_df = pd.read_csv(args.interactions, header=0, index_col=0)
    print(interactions_df.tail(10))

    processed_interactions_df = score_interactions(interactions_df)

    mlflow.log_metric("num_samples", processed_interactions_df.shape[0])
    mlflow.log_metric("num_features", processed_interactions_df.shape[1] - 1)

    processed_interactions_path = args.processed_interactions
    os.makedirs(processed_interactions_path, exist_ok=True)

    processed_interactions = os.path.join(processed_interactions_path, 'processed_interactions.csv')
    processed_interactions_df.to_csv(processed_interactions, index=False)

    # Stop Logging
    mlflow.end_run()


if __name__ == "__main__":
    main()

Overwriting ./scripts/data_prep/data_prep.py


In [9]:
recsys_src_dir = "./scripts/recsys"
os.makedirs(recsys_src_dir, exist_ok=True)

In [10]:
utils_src_dir = "./scripts/recsys/utils"
os.makedirs(utils_src_dir, exist_ok=True)

In [123]:
%%writefile {utils_src_dir}/__init__.py
from .KNNRecommender import KNNRecommender
from .SVDRecommender import SVDRecommender
from .RecommenderSystem import RecommenderSystem

Overwriting ./scripts/recsys/utils/__init__.py


In [124]:
%%writefile {utils_src_dir}/KNNRecommender.py
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds

class KNNRecommender:
    def __init__(self, n_neighbors=100):
        self.n_neighbors = n_neighbors
        self.model = None
        self.user_item_matrix = None

    def fit(self, user_item_matrix):
        self.user_item_matrix = user_item_matrix
        self.model = NearestNeighbors(n_neighbors=self.n_neighbors, metric='cosine')
        self.model.fit(user_item_matrix.values)

    def recommend(self, user_id, n_recommendations=100):
        if self.model is None:
            raise ValueError("Model is not trained yet. Call `fit` with user-item matrix before recommending.")
        
        if user_id not in self.user_item_matrix.index:
            raise ValueError(f"User ID {user_id} not found in the user-item matrix.")
        
        user_vector = self.user_item_matrix.loc[user_id].values.reshape(1, -1)
        distances, indices = self.model.kneighbors(user_vector, n_neighbors=self.n_neighbors + 1)
        similar_user_indices = indices.flatten()[1:]

        similar_users_items = self.user_item_matrix.iloc[similar_user_indices]
        mean_ratings = similar_users_items.mean(axis=0)
        
        user_rated_items = self.user_item_matrix.loc[user_id]
        unrated_items = user_rated_items[user_rated_items == 0].index
        
        recommendations = mean_ratings.loc[unrated_items].sort_values(ascending=False)
        return recommendations.head(n_recommendations).index.tolist()

Overwriting ./scripts/recsys/utils/KNNRecommender.py


In [125]:
%%writefile {utils_src_dir}/SVDRecommender.py
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds

class SVDRecommender():
    def __init__(self, k=10):
        self.k = k
        self.U = None
        self.S = None
        self.Vt = None
        self.user_item_matrix = None

    def fit(self, user_item_matrix):
        self.user_item_matrix = user_item_matrix
        matrix_values = user_item_matrix.fillna(0).values
        U, S, Vt = svds(matrix_values, k=self.k)
        self.U = U
        self.S = np.diag(S)
        self.Vt = Vt

    def recommend(self, user_id, n_recommendations=100):
        if self.U is None or self.S is None or self.Vt is None:
            raise ValueError("Model is not trained yet. Call `fit` with user-item matrix before recommending.")
        
        if user_id not in self.user_item_matrix.index:
            raise ValueError(f"User ID {user_id} not found in the user-item matrix.")
        
        user_index = self.user_item_matrix.index.get_loc(user_id)
        user_vector = self.U[user_index, :]
        predicted_ratings = np.dot(np.dot(user_vector, self.S), self.Vt)
        
        current_ratings = self.user_item_matrix.loc[user_id].values
        predicted_df = pd.DataFrame(predicted_ratings, index=self.user_item_matrix.columns, columns=['predicted'])
        
        interacted_items = set(self.user_item_matrix.columns[current_ratings > 0])
        all_items = set(self.user_item_matrix.columns)
        items_to_predict = all_items - interacted_items
        
        # Convert the set to a list before using as an indexer
        recommendations = predicted_df.loc[list(items_to_predict)].sort_values(by='predicted', ascending=False)
        top_n = recommendations.head(n_recommendations).index.tolist()
        
        return top_n

Overwriting ./scripts/recsys/utils/SVDRecommender.py


In [126]:
%%writefile {utils_src_dir}/RecommenderSystem.py
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds

from utils.KNNRecommender import KNNRecommender
from utils.SVDRecommender import SVDRecommender

class RecommenderSystem:
    def __init__(self, interactions_df, model_type='KNN', **kwargs):
        self.interactions_df = interactions_df
        self.user_item_matrix = None
        self.recommendations_cache = {}

        if model_type == 'KNN':
            self.model = KNNRecommender(**kwargs)
        elif model_type == 'SVD':
            self.model = SVDRecommender(**kwargs)
        else:
            raise ValueError("Unsupported model type. Choose 'KNN' or 'SVD'.")
        
        self.fit()

    def fit(self):
        self.user_item_matrix = self.interactions_df.pivot_table(
            index='user_id', columns='content_id', values='ResolvedScore', aggfunc='sum').fillna(0)
        self.model.fit(self.user_item_matrix)

    def recommend(self, user_id, n_recommendations=100):
        if user_id not in self.recommendations_cache:
            self.recommendations_cache[user_id] = self.model.recommend(user_id, n_recommendations)
        return self.recommendations_cache[user_id][:n_recommendations]

    def get_more_recommendations(self, user_id, start_idx, n_recommendations):
        return self.recommendations_cache[user_id][start_idx:start_idx + n_recommendations]

Overwriting ./scripts/recsys/utils/RecommenderSystem.py


In [127]:
%%writefile {utils_src_dir}/RecommenderSystemWrapper.py
import mlflow.pyfunc
import cloudpickle
import pandas as pd
from utils.RecommenderSystem import RecommenderSystem

class RecommenderSystemWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        with open(context.artifacts["recommender_pickle"], "rb") as f:
            self.model = cloudpickle.load(f)

    def predict(self, context, model_input):
        user_ids = model_input['user_id']
        return user_ids.apply(lambda uid: self.model.recommend(uid, n_recommendations=100))

Overwriting ./scripts/recsys/utils/RecommenderSystemWrapper.py


In [128]:
%%writefile {recsys_src_dir}/recsys.py
import argparse
import os
from datetime import datetime
import pandas as pd
import numpy as np
import mlflow
import tempfile
import cloudpickle

from utils.RecommenderSystem import RecommenderSystem
from utils.RecommenderSystemWrapper import RecommenderSystemWrapper

def select_first_file(path):
    """Selects first file in folder, use under assumption there is only one file in folder
    Args:
        path (str): path to directory or file to choose
    Returns:
        str: full path of selected file
    """
    files = os.listdir(path)
    return os.path.join(path, files[0])

def main():

    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--processed_interactions", 
        type=str, 
        help="path to processed interactions data")
    parser.add_argument(
        "--users", 
        type=str, 
        help="path to users data")
    parser.add_argument("--n_recs", required=False, default=100, type=int)
    parser.add_argument("--registered_model_name", type=str, help="model name")
    parser.add_argument("--model", type=str, help="path to model file")
    parser.add_argument(
        "--all_recommendations", 
        type=str, 
        help="path for storing recommendations")
    args = parser.parse_args()

    mlflow.start_run()
    print(" ".join(f"{k}={v}" for k, v in vars(args).items()))
    print("input data:", args.processed_interactions)

    interactions_df = pd.read_csv(select_first_file(args.processed_interactions))
    print(interactions_df.head())

    users_df = pd.read_csv(args.users, header=0)
    print(users_df.head())

    # Initialize RecommenderSystem
    model_type = 'SVD'  # or 'KNN', depending on your model
    recommender_system = RecommenderSystem(interactions_df, model_type=model_type)
    recommender_system.fit()

    recommendations = {}

    for user_id in users_df['user_id'].unique():
        recommendations[user_id] = recommender_system.recommend(
            user_id, 
            n_recommendations=args.n_recs)

    # Convert to DataFrame after processing all users
    all_recommendations_df = pd.DataFrame(
        list(recommendations.items()), 
        columns=['user_id', 'Recommendations']
    )
    #all_recommendations_df.set_index('user_id', inplace=True)
    print(all_recommendations_df.head())

    with tempfile.TemporaryDirectory() as temp_dir:
        model_path = os.path.join(temp_dir, "recommender.pkl")
        with open(model_path, "wb") as f:
            cloudpickle.dump(RecommenderSystem, f)

    print("Registering the model via MLFlow")
    mlflow.pyfunc.log_model(
        python_model=RecommenderSystemWrapper(),
        code_path=['utils'],
        registered_model_name=args.registered_model_name,
        artifact_path=args.registered_model_name
    )

    # Saving the model to a file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_path = os.path.join(args.model, f"recommender_system_{timestamp}")

    mlflow.pyfunc.save_model(
        python_model=RecommenderSystemWrapper(),
        code_path=['utils'],
        path=model_path,
    )

    # Stop Logging
    mlflow.end_run()

    output_recommendations_path = args.all_recommendations
    os.makedirs(output_recommendations_path, exist_ok=True)

    all_recommendatioins = os.path.join(output_recommendations_path, "all_recommendations.csv")
    all_recommendations_df.to_csv(all_recommendatioins, index=False)

if __name__ == "__main__":
    main()

Overwriting ./scripts/recsys/recsys.py


In [11]:
merged_recs_src_dir = "./scripts/merged_recs"
os.makedirs(merged_recs_src_dir, exist_ok=True)

In [53]:
%%writefile {merged_recs_src_dir}/merged_recs.py
import pandas as pd
import numpy as np
import ast
import os
import argparse
import mlflow

def select_first_file(path):
    """Selects first file in folder, use under assumption there is only one file in folder
    Args:
        path (str): path to directory or file to choose
    Returns:
        str: full path of selected file
    """
    files = os.listdir(path)
    return os.path.join(path, files[0])

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--all_recommendations", 
        type=str, 
        help="path to the recommendations")
    parser.add_argument(
        "--content", 
        type=str, 
        help="path to content data")
    parser.add_argument(
        "--merged_recommendations", 
        type=str, 
        help="path for storing merged recommendations")
    args = parser.parse_args()

    mlflow.start_run()
    print(" ".join(f"{k}={v}" for k, v in vars(args).items()))
    print("input data:", args.all_recommendations)

    all_recommendations_df = pd.read_csv(select_first_file(args.all_recommendations), index_col=0, header=0).reset_index(drop=False)
    print(all_recommendations_df.head())

    content_df = pd.read_csv(args.content, header=0)
    print(content_df.head())

    all_recommendations_df['Recommendations'] = all_recommendations_df['Recommendations'].apply(ast.literal_eval)
    all_recommendations_df = all_recommendations_df.explode("Recommendations")
    #recommendations_exploded = all_recommendations_df.explode('Recommendations', ignore_index=True)
    all_recommendations_df = all_recommendations_df.rename(
        columns={'Recommendations': 'content_id'})

    merged_recommendations_df = pd.merge(
        all_recommendations_df, 
        content_df, 
        on='content_id', 
        how='left')

    print(merged_recommendations_df.columns)
    #merged_recommendations_df = merged_recommendations_df.dropna()
    print(merged_recommendations_df.isnull().sum().any())
    print(len(merged_recommendations_df))

    mlflow.log_metric("num_samples", merged_recommendations_df.shape[0])
    mlflow.log_metric("num_features", merged_recommendations_df.shape[1] - 1)

    merged_recommendations_path = args.merged_recommendations
    os.makedirs(merged_recommendations_path, exist_ok=True)

    merged_recommendations = os.path.join(merged_recommendations_path, "merged_recommendations.csv")
    merged_recommendations_df.to_csv(merged_recommendations, index=False)

    mlflow.end_run()

if __name__ == "__main__":
    main()


Overwriting ./scripts/merged_recs/merged_recs.py


In [12]:
ranked_recs_src_dir = "./scripts/ranked_recs"
os.makedirs(ranked_recs_src_dir, exist_ok=True)

In [13]:
%%writefile {ranked_recs_src_dir}/ranked_recs.py
import pandas as pd
import numpy as np
import ast
import os
import argparse
import mlflow

def downrank_pc_ndc_content(df, remove_or_rerank='rerank', apply_to_all=False):
    results = pd.DataFrame(columns=df.columns)
    unique_users = df['user_id'].unique()

    for user in unique_users:
        user_recs = df[df['user_id'] == user]
        rows_to_keep = []
        rows_to_rerank = []

        for i in range(len(user_recs)):
            harm_class = user_recs.iloc[i].get('harms_classification')
            is_child = user_recs.iloc[i].get('is_child')

            target = harm_class in ['PC', 'NDC']
            condition = apply_to_all or (is_child is True)

            if target and condition:
                if remove_or_rerank == 'remove':
                    continue
                elif remove_or_rerank == 'rerank':
                    rows_to_rerank.append(user_recs.iloc[i])
            else:
                rows_to_keep.append(user_recs.iloc[i])

        updated_recs = pd.DataFrame(rows_to_keep + rows_to_rerank, columns=df.columns)
        results = pd.concat([results, updated_recs], ignore_index=True)

    return results

def filter_ppc_content(df, remove_or_rerank='remove', apply_to_all=True):
    results = pd.DataFrame(columns=df.columns)
    unique_users = df['user_id'].unique()

    for user in unique_users:
        user_recs = df[df['user_id'] == user]
        rows_to_keep = []
        rows_to_rerank = []

        for i in range(len(user_recs)):
            harm_class = user_recs.iloc[i].get('HarmCategoryClassification')
            is_child = user_recs.iloc[i].get('is_child')

            target = harm_class == 'PPC'
            condition = apply_to_all or (is_child is True)

            if target and condition:
                if remove_or_rerank == 'remove':
                    continue
                elif remove_or_rerank == 'rerank':
                    rows_to_rerank.append(user_recs.iloc[i])
            else:
                rows_to_keep.append(user_recs.iloc[i])

        updated_recs = pd.DataFrame(rows_to_keep + rows_to_rerank, columns=df.columns)
        results = pd.concat([results, updated_recs], ignore_index=True)

    return results

#TODO this has to be fixed once liked sub categories are recorded for users
def uprank_liked_categories(df):
    results = pd.DataFrame(columns=df.columns)
    unique_users = df['user_id'].unique()

    for user in unique_users:
        user_recs = df[df['user_id'] == user]
        liked_rows = []
        other_rows = []

        for i in range(len(user_recs)):
            article_category = user_recs.iloc[i]['sub_category']
            user_liked_cat = user_recs.iloc[i]['liked_cat']

            if article_category == user_liked_cat:
                liked_rows.append(user_recs.iloc[i])
            else:
                other_rows.append(user_recs.iloc[i])

        rearranged_recs = pd.DataFrame(liked_rows + other_rows, columns=df.columns)
        results = pd.concat([results, rearranged_recs], ignore_index=True)

    return results

def select_first_file(path):
    files = os.listdir(path)
    return os.path.join(path, files[0])

def blockwise_shuffle(df, block_size=5, seed=None):
    if seed is not None:
        np.random.seed(seed)

    num_blocks = len(df) // block_size
    blocks = [df.iloc[i * block_size : (i + 1) * block_size] for i in range(num_blocks)]
    np.random.shuffle(blocks)
    shuffled_df = pd.concat(blocks, ignore_index=True)

    return shuffled_df

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--merged_recommendations", type=str, help="path to the merged recommendations")
    parser.add_argument("--ranked_recommendations", type=str, help="path for storing ranked recommendations")
    args = parser.parse_args()

    mlflow.start_run()
    print(" ".join(f"{k}={v}" for k, v in vars(args).items()))
    print("input data:", args.merged_recommendations)

    merged_recommendations_df = pd.read_csv(select_first_file(args.merged_recommendations))
    print(merged_recommendations_df.head())

    # Apply filtering and re-ranking
    df = downrank_pc_ndc_content(merged_recommendations_df)  # PC + NDC for children only
    df = filter_ppc_content(df)                              # PPC for everyone
    #df = uprank_liked_categories(df)                         # Liked categories to top

    print('These are the reranked_recommendations')
    print(df.head())
    print(df.columns)

    mlflow.log_metric("num_samples", df.shape[0])
    mlflow.log_metric("num_features", df.shape[1] - 1)

    output_dir = args.ranked_recommendations
    os.makedirs(output_dir, exist_ok=True)
    df.to_csv(os.path.join(output_dir, "ranked_recommendations.csv"), index=False)

    mlflow.end_run()

if __name__ == "__main__":
    main()

Overwriting ./scripts/ranked_recs/ranked_recs.py


In [14]:
from azure.ai.ml import command
from azure.ai.ml import Input, Output
from azure.ai.ml.constants import AssetTypes, InputOutputModes

data_prep_component = command(
    name="data_prep_recsys_app",
    display_name="Process interaction data",
    description="encode interaction types",
    is_deterministic=False,
    inputs={
        "interactions": Input(type=AssetTypes.URI_FILE),
    },
    outputs=dict(
        processed_interactions=Output(
            type="uri_folder", 
            mode="rw_mount",
            path=f"azureml://datastores/{DATASTORE_NAME}/paths/processed_interactions/"),
    ),
    # The source folder of the component
    code=data_prep_src_dir,
    command="""python data_prep.py \
            --interactions ${{inputs.interactions}} \
            --processed_interactions ${{outputs.processed_interactions}} \
            """,
    environment=f"{pipeline_job_env.name}:{pipeline_job_env.version}",
)

In [15]:
%%writefile {recsys_src_dir}/recsys.yml
# <component>
name: train_recsys
display_name: Train Recommender System
# version: 1 # Not specifying a version will automatically update the version
type: command
is_deterministic: False
inputs:
  processed_interactions: 
    type: uri_folder
  users: 
    type: uri_folder
  n_recs:
    type: integer     
  registered_model_name:
    type: string
outputs:
  all_recommendations:
    type: uri_folder
  model:
    type: uri_folder
code: .
environment:
  azureml:/subscriptions/42f44488-1a2d-4230-b7df-e002ba021ab6/resourceGroups/trustandsafety-cv-sandbox-rg/providers/Microsoft.MachineLearningServices/workspaces/aml-trustandsafety-cv/environments/recsys_app/versions/0.2.4
command: >-
  python recsys.py 
  --processed_interactions ${{inputs.processed_interactions}} 
  --users ${{inputs.users}} 
  --n_recs ${{inputs.n_recs}}
  --registered_model_name ${{inputs.registered_model_name}} 
  --all_recommendations ${{outputs.all_recommendations}}
  --model ${{outputs.model}}
# </component>

Overwriting ./scripts/recsys/recsys.yml


In [16]:
# importing the Component Package
from azure.ai.ml import load_component

# Loading the component from the yml file
recsys_component = load_component(source=os.path.join(recsys_src_dir, "recsys.yml"))

# Now we register the component to the workspace
recsys_component = ml_client.create_or_update(recsys_component)

# Create (register) the component in your workspace
print(
    f"Component {recsys_component.name} with Version {recsys_component.version} is registered"
)

Component train_recsys with Version 2025-05-22-12-18-58-6446090 is registered


In [17]:
merged_recs_component = command(
    name="merge_recs_recsys_app",
    display_name="Merge recommendations",
    description="merge recommendations with content data",
    is_deterministic=False,
    inputs={
        "all_recommendations": Input(type='uri_folder'),
        "content": Input(type=AssetTypes.URI_FILE),
    },
    outputs=dict(
        merged_recommendations=Output(
            type="uri_folder", 
            mode="rw_mount",
            path=f"azureml://datastores/{DATASTORE_NAME}/paths/merged_recommendations/"),
    ),
    # The source folder of the component
    code=merged_recs_src_dir,
    command="""python merged_recs.py \
            --all_recommendations ${{inputs.all_recommendations}} \
            --content ${{inputs.content}} \
            --merged_recommendations ${{outputs.merged_recommendations}} \
            """,
    environment=f"{pipeline_job_env.name}:{pipeline_job_env.version}",
)

In [18]:
ranked_recs_component = command(
    name="rank_recs_recsys_app",
    display_name="Ranking algorithms",
    description="rank recommendations",
    is_deterministic=False,
    inputs={
        "merged_recommendations": Input(type='uri_folder'),
    },
    outputs=dict(
        ranked_recommendations=Output(
            type="uri_folder", 
            mode="rw_mount",
            path=f"azureml://datastores/{DATASTORE_NAME}/paths/ranked_recommendations/"),
    ),
    # The source folder of the component
    code=ranked_recs_src_dir,
    command="""python ranked_recs.py \
            --merged_recommendations ${{inputs.merged_recommendations}} \
            --ranked_recommendations ${{outputs.ranked_recommendations}} \
            """,
    environment=f"{pipeline_job_env.name}:{pipeline_job_env.version}",
)

In [19]:
# the dsl decorator tells the sdk that we are defining an Azure Machine Learning pipeline
from azure.ai.ml import dsl, Input, Output

@dsl.pipeline(
    compute="cpu-cluster",
    description="E2E rec sys pipeline",
)
def recsys_app_pipeline(
    pipeline_job_data_input: Input,
    pipeline_job_users: Input,
    pipeline_job_n_recs: int,
    pipeline_job_registered_model_name: str,
    pipeline_job_content: Input
    ):

    # using data_prep_function like a python call with its own inputs
    data_prep_job = data_prep_component(
        interactions=pipeline_job_data_input
    )

    recsys_job = recsys_component(
        processed_interactions=data_prep_job.outputs.processed_interactions,
        users=pipeline_job_users,
        n_recs=pipeline_job_n_recs,
        registered_model_name=pipeline_job_registered_model_name
    )

    merge_recs_job = merged_recs_component(
        all_recommendations=recsys_job.outputs.all_recommendations,
        content=pipeline_job_content
    )

    rank_recs_job = ranked_recs_component(
        merged_recommendations=merge_recs_job.outputs.merged_recommendations
    )

    # direct the outputs to the blob storage here 
    recsys_job.outputs.all_recommendations = Output(
        type="uri_folder",
        path=f"azureml://datastores/{DATASTORE_NAME}/paths/all_recommendations/",
        mode="rw_mount"
    )

    recsys_job.outputs.model = Output(
        type="uri_folder",
        path=f"azureml://datastores/{DATASTORE_NAME}/paths/recsys_model/",
        mode="rw_mount"
    )

    merge_recs_job.outputs.merged_recommendations = Output(
        type="uri_folder",
        path=f"azureml://datastores/{DATASTORE_NAME}/paths/merged_recommendations/",
        mode='rw_mount'
    )

    rank_recs_job.outputs_ranked_recommendations = Output(
        type='uri_folder',
        path=f"azureml://datastores/{DATASTORE_NAME}/paths/ranked_recommendations/",
        mode='rw_mount'
    )

    # a pipeline returns a dictionary of outputs
    # keys will code for the pipeline output identifier
    return {
        "pipeline_job_processed_interactions": data_prep_job.outputs.processed_interactions,
        "pipeline_job_all_recommendations": recsys_job.outputs.all_recommendations,
        "pipeline_job_merged_recommendations": merge_recs_job.outputs.merged_recommendations,
        "pipeline_job_ranked_recommendations": rank_recs_job.outputs.ranked_recommendations
    }

In [20]:
# initial data
interactions_path = f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}/paths/interactions/interactions.csv"
users_path = f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}/paths/users/users.csv"
content_path = f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}/paths/content/content.csv"

In [21]:
registered_model_name = "recsys_app"

# instantiate the pipeline with the parameters of our choice
pipeline = recsys_app_pipeline(
    pipeline_job_data_input=Input(
        type="uri_file", 
        path=interactions_path,
        mode=InputOutputModes.RO_MOUNT
    ),
    pipeline_job_users=Input(
        type="uri_file",
        path=users_path,
        mode=InputOutputModes.RO_MOUNT
    ),
    pipeline_job_n_recs=100, # change the recommendations
    pipeline_job_registered_model_name="recsys_app",
    pipeline_job_content=Input(
        type='uri_file',
        path=content_path,
        mode=InputOutputModes.RO_MOUNT
    )
)

pipeline.settings.default_datastore = "recsys_database"

In [22]:
pipeline_job = ml_client.jobs.create_or_update(
    pipeline,
    experiment_name="RecSys_App",
    force_rerun=True
)
ml_client.jobs.stream(pipeline_job.name)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Warnings: [jobs.rank_recs_job.outputs_

RunId: polite_worm_4y3xmpn3jl
Web View: https://ml.azure.com/runs/polite_worm_4y3xmpn3jl?wsid=/subscriptions/42f44488-1a2d-4230-b7df-e002ba021ab6/resourcegroups/trustandsafety-cv-sandbox-rg/workspaces/aml-trustandsafety-cv

Streaming logs/azureml/executionlogs.txt

[2025-05-22 12:19:23Z] Submitting 1 runs, first five are: dfaf4632:9ece58e5-5ff4-47f8-b26b-0dd4d0c15465
[2025-05-22 12:28:11Z] Completing processing run id 9ece58e5-5ff4-47f8-b26b-0dd4d0c15465.
[2025-05-22 12:28:12Z] Submitting 1 runs, first five are: da5ba64c:99b3650d-5ee7-428f-b732-68fc152b1f72
[2025-05-22 12:29:04Z] Completing processing run id 99b3650d-5ee7-428f-b732-68fc152b1f72.
[2025-05-22 12:29:04Z] Submitting 1 runs, first five are: 8ff22bf3:5161f689-d40b-4e60-bda7-3089a9e111d5
[2025-05-22 12:29:51Z] Completing processing run id 5161f689-d40b-4e60-bda7-3089a9e111d5.
[2025-05-22 12:29:52Z] Submitting 1 runs, first five are: 7f541eab:06959772-e4a3-429c-a687-f7e6fdbd0484
[2025-05-22 12:30:45Z] Completing processing run

In [23]:
# schedule the pipeline triggering - outputs cleared
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Schedule, CronTrigger, JobSchedule
from datetime import datetime

schedule_name = "recsys-retrain-schedule"
schedule_start_time = datetime.utcnow()

cron_trigger = CronTrigger(
    expression="0 0 * * *",  # Every day at midnight
    start_time=schedule_start_time,
    time_zone="UTC"
)

job_schedule = JobSchedule(
    name=schedule_name, trigger=cron_trigger, create_job=pipeline
)

job_schedule = ml_client.schedules.begin_create_or_update(schedule=job_schedule).result()
print(job_schedule)

Warnings: [jobs.rank_recs_job.outputs_ranked_recommendations: Unknown field.]
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


..name: recsys-retrain-schedule
create_job:
  description: E2E rec sys pipeline
  display_name: recsys_app_pipeline
  status: NotStarted
  experiment_name: RecSys_App
  properties:
    mlflow.source.git.repoURL: https://github.com/ofcomuk/RecSys_App
    mlflow.source.git.branch: main
    mlflow.source.git.commit: ab26dcb9d5f1c32cf860fc1ec3a50d02e1631884
    azureml.git.dirty: 'True'
  compute: azureml:cpu-cluster
  type: pipeline
  settings:
    default_datastore: azureml:/subscriptions/42f44488-1a2d-4230-b7df-e002ba021ab6/resourceGroups/trustandsafety-cv-sandbox-rg/providers/Microsoft.MachineLearningServices/workspaces/aml-trustandsafety-cv/datastores/recsys_database
  inputs:
    pipeline_job_data_input:
      mode: ro_mount
      type: uri_file
      path: azureml://subscriptions/42f44488-1a2d-4230-b7df-e002ba021ab6/resourcegroups/trustandsafety-cv-sandbox-rg/workspaces/aml-trustandsafety-cv/datastores/recsys_database/paths/interactions/interactions.csv
    pipeline_job_users:
     

In [38]:
# endpoint created

#import uuid
#from azure.ai.ml.entities import BatchEndpoint

# Create a unique name for the endpoint
#batch_endpoint_name = "recsys-app-deployment" + str(uuid.uuid4())[:8] #recsys-app-deploymente9e1396d

#endpoint = BatchEndpoint(
#    name=batch_endpoint_name,
#    description="Endpoint for rec sys pipeline deployments",
#)

# create the batch endpoint
# expect the endpoint to take approximately 2 minutes.

#endpoint = ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

In [ ]:
# to check if batch endpoint is created, output is cleared

#endpoint = ml_client.batch_endpoints.get(name="recsys-app-deploymente9e1396d") #batch_endpoint_name)
#print(endpoint)

In [ ]:
# append pipeline to the endpoint - re-run this if pipeline is changed

#recsys_pipeline_component = ml_client.components.create_or_update(recsys_app_pipeline)

In [87]:
# create a deployment - re-run this if pipeline is changed

#from azure.ai.ml.entities import PipelineComponentBatchDeployment

#deployment = PipelineComponentBatchDeployment(
#    name="recsys-app-pipeline-dep",
#    description="A deployment that preprocess, recommends, and ranks",
#    endpoint_name=endpoint.name,
#    component=recsys_pipeline_component,
#    settings={
#        "continue_on_step_failure": False, 
#        "default_compute": "cpu-cluster",
#        "default_datastore": "recsys_database"},
#)

# output is cleared, check under endpoint 
#ml_client.batch_deployments.begin_create_or_update(deployment).result()

# output is cleared
#endpoint.defaults.deployment_name = deployment.name
#ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

#print(f"The default deployment is {endpoint.defaults.deployment_name}") # recsys-app-pipeline-dep

In [ ]:
# trigger the pipeline via sdk

#ml_client = MLClient(
#    credential=credential,
#    subscription_id=SUBSCRIPTION,
#    resource_group_name=RESOURCE_GROUP,
#    workspace_name=WS_NAME,
#)

#interactions_path = f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}/paths/interactions/interactions.csv"
#users_path = f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}/paths/users/users.csv"
#content_path = f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}/paths/content/content.csv"

#inputs = {
#    "pipeline_job_data_input":  Input(type="uri_file", path=interactions_path),
#    "pipeline_job_users": Input(type="uri_file", path=users_path),
#    "pipeline_job_n_recs":Input(type='integer', default=5),
#    "pipeline_job_registered_model_name": Input(type='string', default="recsys_app_test"),
#    "pipeline_job_content": Input(type='uri_file',path=content_path)
#}

#job = ml_client.batch_endpoints.invoke(
#    endpoint_name='recsys-app-deploymente9e1396d',
#    inputs=inputs
#)

#while True:
#    status = ml_client.jobs.get(job.name).status

#    if status in ["Completed", "Failed", "Canceled"]:
#        break
#    time.sleep(5)

#if status == "Completed":
#    print("Pipeline run completed successfully!")

#ml_client.jobs.download(
#    name=job.name, download_path=LOCAL_DOWNLOAD_PATH, output_name="pipeline_job_ranked_recommendations"
#)

#output_file = os.path.join(
#    f"{LOCAL_DOWNLOAD_PATH}/named-outputs/pipeline_job_ranked_recommendations", 
#    "ranked_recommendations.csv"
#    )

In [105]:
#trigger the pipeline via rest, but doesn't seem to work

#from azureml.core.authentication import InteractiveLoginAuthentication
#from azure.identity import ManagedIdentityCredential

#interactive_auth = InteractiveLoginAuthentication()

#credential = ManagedIdentityCredential()
#token = credential.get_token("https://ml.azure.com/.default")

#endpoint_url = "https://recsys-app-deploymente9e1396d.uksouth.inference.ml.azure.com/jobs"

#headers = {
#    "Authorization": f"Bearer {token.token}",
#    "Content-Type": "application/json"
#}

#request_body = {
#    "properties": {
#        "inputData": {
#            "pipeline_job_data_input": {
#                "jobInputType": "UriFile",
#                "uri": interactions_path
#                },
#            "pipeline_job_users": {
#                "jobInputType": "UriFile",
#                "uri": users_path
#                },
#            "pipeline_job_content": {
#                "jobInputType": "UriFile",
#                "uri": content_path
#                },
#            "pipeline_job_n_recs": {
#                "jobInputType": "literal",
#                "value": 5
#                },
#            "pipeline_job_registered_model_name": {
#                "jobInputType": "literal",
#                "value": "recsys_model_test_rest"
#                }
#        }
#    }
#}

#import requests
#response = requests.post(endpoint_url, headers=headers, json=request_body)

#response.status_code

#job_id = response.json()['name']
#job_status = requests.get(f"{endpoint_url}/{job_id}", headers=headers)
#job_status.json()['properties']['status']